# **Text Generation with Recurrent Neural Networks (RNN)**

**1. Import Necessary Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import collections
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Setting random seeds
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


**2. Web Scraping the Data**

In [2]:
url = "https://www.gutenberg.org/cache/epub/84/pg84-images.html"
print("Downloading text...")
response = requests.get(url)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, 'html.parser')
raw_text = soup.get_text()

start_marker = "Letter 1"
end_marker = "THE END"

try:
    start_idx = raw_text.find(start_marker)
    end_idx = raw_text.find(end_marker)
    if start_idx != -1 and end_idx != -1:
        text_data = raw_text[start_idx:end_idx + len(end_marker)]
    else:
        text_data = raw_text
except:
    text_data = raw_text

print(f"Text length: {len(text_data)} characters.")

Text length: 446307 characters.


**3. Data Preprocessing and Tokenization**

In [3]:
tokens = re.findall(r'\b\w+\b', text_data.lower())
vocab = sorted(list(set(tokens)))
vocab_size = len(vocab)

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 7309


**4. Preparing Sequences for RNN**

In [4]:
window_size = 100
seq_length = window_size - 1
inputs = []
targets = []
step = 3

token_indices = [word_to_idx[word] for word in tokens]

for i in range(0, len(token_indices) - window_size, step):
    inputs.append(token_indices[i : i + seq_length])
    targets.append(token_indices[i + seq_length])

print(f"Number of sequences: {len(inputs)}")

Number of sequences: 26147


**5. Custom Dataset and DataLoader**

In [5]:
class RNNTextDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.LongTensor(inputs)
        self.targets = torch.LongTensor(targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

BATCH_SIZE = 64
dataset = RNNTextDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("DataLoader ready.")

DataLoader ready.


**6. RNN Model Implementation**

In [6]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=1):
        super(SimpleRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 2. RNN Layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        # 3. Fully Connected Layer
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embeds = self.embedding(x)
        out, hidden = self.rnn(embeds, hidden)

        out = out[:, -1, :]
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)

**7. Hyperparameters and Model Initialization**

In [7]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
LEARNING_RATE = 0.005
EPOCHS = 20

model = SimpleRNN(vocab_size, EMBEDDING_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Model moved to GPU.")

Model moved to GPU.


**8. Training Loop**

In [8]:
print(f"Starting training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        hidden = model.init_hidden(BATCH_SIZE)
        optimizer.zero_grad()

        output, hidden = model(x_batch, hidden)
        loss = criterion(output, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(dataloader):.4f}")

print("Training finished.")

Starting training on cuda...
Epoch 1/20 | Loss: 7.9125
Epoch 2/20 | Loss: 6.9178
Epoch 3/20 | Loss: 6.5403
Epoch 4/20 | Loss: 6.2557
Epoch 5/20 | Loss: 5.9603
Epoch 6/20 | Loss: 5.2479
Epoch 7/20 | Loss: 5.1184
Epoch 8/20 | Loss: 5.4911
Epoch 9/20 | Loss: 6.0212
Epoch 10/20 | Loss: 5.9929
Epoch 11/20 | Loss: 5.9916
Epoch 12/20 | Loss: 7.6185
Epoch 13/20 | Loss: 7.2843
Epoch 14/20 | Loss: 7.2461
Epoch 15/20 | Loss: 8.7158
Epoch 16/20 | Loss: 6.8169
Epoch 17/20 | Loss: 6.7189
Epoch 18/20 | Loss: 7.3622
Epoch 19/20 | Loss: 7.0379
Epoch 20/20 | Loss: 8.3918
Training finished.


**9. Text Generation**

In [9]:
def generate_text_gpu(model, seed_text, next_words=100, temperature=1.0):
    model.eval()

    words = re.findall(r'\b\w+\b', seed_text.lower())
    current_seq = [word_to_idx.get(w, 0) for w in words]

    x_in = torch.LongTensor([current_seq]).to(device)

    # Init hidden on GPU
    hidden = model.init_hidden(1)

    generated = words[:]

    with torch.no_grad():
        _, hidden = model(x_in, hidden)

        last_word_idx = current_seq[-1]

        for _ in range(next_words):
            # 2. Input only the last word
            x_in = torch.LongTensor([[last_word_idx]]).to(device)

            output, hidden = model(x_in, hidden)

            probs = torch.softmax(output / temperature, dim=1).cpu().numpy()[0]

            pred_idx = np.random.choice(len(probs), p=probs)
            pred_word = idx_to_word[pred_idx]

            generated.append(pred_word)
            last_word_idx = pred_idx

    return " ".join(generated)

# Testing
start_index = random.randint(0, len(tokens) - 150)
seed_text = " ".join(tokens[start_index : start_index + 20])

print(f"--- Seed ---\n{seed_text}\n")
print(f"--- Generated (GPU) ---\n")
print(generate_text_gpu(model, seed_text, next_words=100, temperature=0.8))

--- Seed ---
of the cottagers now opened new wonders to me while i listened to the instructions which felix bestowed upon the

--- Generated (GPU) ---

of the cottagers now opened new wonders to me while i listened to the instructions which felix bestowed upon the imagination well agrippa lausanne endless endless broad departed one through a warmth that archive accents female greece ceremony regularly had taken earth here gesture fearful raft especially fixed stimulated claims him itself and valued you will consecrate countenance revenge hear ecstasy st awe turning persuaded esteem into not land curiosity substance sight updated society directly subscribed devils fainted asked endured departed answer with agrippa lausanne gestures besieged prevented content would would not a strange person inhabitants mate avowal superior deer slow nothing would bring for mind of mind performances fearful gestures touched st particular endured with my 11 animation calmness
